# Batch Size Ablation Study (SSL only)

This notebook trains SimCLR with different batch sizes and compares the learning curves.

**Before running:** `Runtime → Change runtime type → T4 GPU`

### Why batch size matters specifically for SSL

In SimCLR, every image in the batch is used as a **negative sample** for every other image.
With batch size N, each image sees **2N − 2 negatives** during training:

| Batch size | Negatives per image |
|---|---|
| 512 | 1022 |
| 256 | 510 |
| 128 | 254 |
| 64 | 126 |
| 32 | 62 |

More negatives = harder contrastive task = stronger learning signal = better representations.
This effect is unique to contrastive SSL — it does not apply to supervised learning.

### What you will get
- Overlaid **loss curves** for all batch sizes
- Overlaid **kNN accuracy curves** for all batch sizes
- Summary table of final linear probing accuracy per batch size

## 1. Setup

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.optim import Adam
from torch.utils.data import DataLoader
import torchvision.models as models
from torchvision import datasets, transforms
import matplotlib.pyplot as plt
import os

if torch.cuda.is_available():
    DEVICE = torch.device('cuda')
    vram = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"GPU: {torch.cuda.get_device_name(0)}  ({vram:.1f} GB VRAM)")
else:
    DEVICE = torch.device('cpu')
    print("WARNING: No GPU. Go to Runtime → Change runtime type → GPU")

print(f"PyTorch: {torch.__version__}")

In [ ]:
USE_DRIVE = False   # Set True to save results to Google Drive

if USE_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    BASE_DIR = '/content/drive/MyDrive/AI_HW2/ablation_batchsize'
else:
    BASE_DIR = '/content/ablation_batchsize'

os.makedirs(f'{BASE_DIR}/checkpoints', exist_ok=True)
os.makedirs(f'{BASE_DIR}/results',     exist_ok=True)
print(f"Working directory: {BASE_DIR}")

In [ ]:
import sys
from datetime import datetime

class _Tee:
    """Mirrors all print() output to both the notebook and a persistent log file."""
    def __init__(self, filepath):
        self._file   = open(filepath, 'a')
        self._stdout = sys.stdout
    def write(self, msg):
        self._stdout.write(msg)
        self._file.write(msg)
    def flush(self):
        self._stdout.flush()
        self._file.flush()
    def __enter__(self):
        sys.stdout = self
        return self
    def __exit__(self, *args):
        sys.stdout = self._stdout
        self._file.close()

LOG_FILE = f'{BASE_DIR}/results/training_log.txt'

with open(LOG_FILE, 'w') as f:
    f.write(f"Batch Size Ablation Log\n")
    f.write(f"Started : {datetime.now().strftime('%Y-%m-%d %H:%M:%S')}\n")
    f.write("=" * 60 + "\n\n")

print(f"Log file ready: {LOG_FILE}")

## 2. Config

**Edit `BATCH_SIZES`** to choose which values to compare.  
Start from the largest your GPU can handle and go down.

A T4 (16 GB) can handle up to **512** comfortably. If you get an out-of-memory error, remove the largest value.

In [ ]:
# -----------------------------------------------------------------------
# Batch sizes to compare — this is the variable being ablated
# -----------------------------------------------------------------------
BATCH_SIZES = [512, 256, 128, 64, 32]

# -----------------------------------------------------------------------
# Fixed settings (same for all runs so the comparison is fair)
# -----------------------------------------------------------------------
DATA_DIR    = '/content/data'
NUM_CLASSES = 10
NUM_WORKERS = 2

SSL_EPOCHS       = 200     # Use 100 for a faster run
TEMPERATURE      = 0.5     # Fixed at default — we are only varying batch size
SSL_LR           = 3e-4
SSL_WEIGHT_DECAY = 1e-6

PROJECTOR_HIDDEN_DIM = 512
PROJECTOR_OUT_DIM    = 128

KNN_K        = 20
KNN_INTERVAL = 5
KNN_TEMP     = 0.1

LP_EPOCHS       = 100
LP_LR           = 1e-3
LP_WEIGHT_DECAY = 1e-6

# Eval loader uses the largest batch size for speed (batch size doesn't affect eval results)
EVAL_BATCH_SIZE = 512

print(f"Will train {len(BATCH_SIZES)} models: batch sizes = {BATCH_SIZES}")
print(f"Each run: {SSL_EPOCHS} epochs, temperature {TEMPERATURE}")

# Estimate negatives per image for each batch size
print(f"\nNegatives per image:")
for bs in BATCH_SIZES:
    print(f"  batch {bs:>4}  →  {2*bs - 2:>5} negatives")

## 3. Model, Loss & Evaluation Definitions

Self-contained — runs independently from the main notebook.

In [ ]:
CIFAR10_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR10_STD  = (0.2023, 0.1994, 0.2010)

class SimCLRTransform:
    def __init__(self, image_size=32):
        self.augment = transforms.Compose([
            transforms.RandomResizedCrop(image_size, scale=(0.2, 1.0)),
            transforms.RandomHorizontalFlip(p=0.5),
            transforms.RandomApply([
                transforms.ColorJitter(brightness=0.4, contrast=0.4,
                                       saturation=0.4, hue=0.1)
            ], p=0.8),
            transforms.RandomGrayscale(p=0.2),
            transforms.ToTensor(),
            transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
        ])
    def __call__(self, image):
        return self.augment(image), self.augment(image)

EVAL_TRANSFORM = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR10_MEAN, std=CIFAR10_STD),
])

def get_ssl_loader(batch_size):
    dataset = datasets.CIFAR10(root=DATA_DIR, train=True, download=True,
                                transform=SimCLRTransform())
    return DataLoader(dataset, batch_size=batch_size, shuffle=True,
                      num_workers=NUM_WORKERS, drop_last=True, pin_memory=True)

def get_eval_loaders(batch_size):
    train_set = datasets.CIFAR10(root=DATA_DIR, train=True,  download=True, transform=EVAL_TRANSFORM)
    test_set  = datasets.CIFAR10(root=DATA_DIR, train=False, download=True, transform=EVAL_TRANSFORM)
    return (DataLoader(train_set, batch_size=batch_size, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True),
            DataLoader(test_set,  batch_size=batch_size, shuffle=False,
                       num_workers=NUM_WORKERS, pin_memory=True))

print("Dataset defined.")

In [ ]:
class ModifiedResNet18(nn.Module):
    def __init__(self):
        super().__init__()
        resnet = models.resnet18(weights=None)
        resnet.conv1   = nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False)
        resnet.maxpool = nn.Identity()
        self.encoder   = nn.Sequential(*list(resnet.children())[:-1])
        self.output_dim = 512
    def forward(self, x):
        return torch.flatten(self.encoder(x), start_dim=1)

class ProjectorHead(nn.Module):
    def __init__(self, in_dim=512, hidden_dim=512, out_dim=128):
        super().__init__()
        self.mlp = nn.Sequential(
            nn.Linear(in_dim, hidden_dim), nn.ReLU(inplace=True),
            nn.Linear(hidden_dim, out_dim),
        )
    def forward(self, x): return self.mlp(x)

class SimCLRModel(nn.Module):
    def __init__(self):
        super().__init__()
        self.backbone  = ModifiedResNet18()
        self.projector = ProjectorHead(512, PROJECTOR_HIDDEN_DIM, PROJECTOR_OUT_DIM)
    def forward(self, x):
        h = self.backbone(x)
        return h, self.projector(h)
    def encode(self, x):
        return self.backbone(x)

class NTXentLoss(nn.Module):
    def __init__(self, temperature):
        super().__init__()
        self.temperature = temperature
    def forward(self, z_i, z_j):
        N   = z_i.shape[0]
        z   = torch.cat([F.normalize(z_i, dim=1), F.normalize(z_j, dim=1)], dim=0)
        sim = torch.mm(z, z.T) / self.temperature
        labels = torch.cat([torch.arange(N, device=z_i.device) + N,
                             torch.arange(N, device=z_i.device)])
        sim = sim.masked_fill(torch.eye(2*N, dtype=torch.bool, device=z_i.device), float('-inf'))
        return F.cross_entropy(sim, labels)

print("Model and loss defined.")

In [ ]:
@torch.no_grad()
def extract_features(model, loader):
    model.eval()
    feats, labs = [], []
    for images, labels in loader:
        feats.append(model.encode(images.to(DEVICE)).cpu())
        labs.append(labels)
    return torch.cat(feats), torch.cat(labs)

@torch.no_grad()
def knn_monitor(model, train_loader, test_loader):
    train_feats, train_labels = extract_features(model, train_loader)
    test_feats,  test_labels  = extract_features(model, test_loader)
    train_feats  = F.normalize(train_feats, dim=1).to(DEVICE)
    test_feats   = F.normalize(test_feats,  dim=1).to(DEVICE)
    train_labels = train_labels.to(DEVICE)
    correct = total = 0
    for start in range(0, len(test_feats), 512):
        end           = min(start + 512, len(test_feats))
        batch_feats   = test_feats[start:end]
        batch_labels  = test_labels[start:end].to(DEVICE)
        sim           = torch.mm(batch_feats, train_feats.T) / KNN_TEMP
        topk_vals, topk_idx = sim.topk(KNN_K, dim=1)
        weights       = F.softmax(topk_vals, dim=1)
        neighbor_labs = train_labels[topk_idx]
        votes         = torch.zeros(end - start, NUM_CLASSES, device=DEVICE)
        votes.scatter_add_(1, neighbor_labs, weights)
        correct += (votes.argmax(1) == batch_labels).sum().item()
        total   += len(batch_labels)
    return correct / total

def linear_probing(model, train_loader, test_loader):
    for p in model.backbone.parameters(): p.requires_grad = False
    model.eval()
    linear    = nn.Linear(512, NUM_CLASSES).to(DEVICE)
    optimizer = Adam(linear.parameters(), lr=LP_LR, weight_decay=LP_WEIGHT_DECAY)
    loss_fn   = nn.CrossEntropyLoss()
    for _ in range(LP_EPOCHS):
        linear.train()
        for images, labels in train_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            with torch.no_grad(): features = model.encode(images)
            loss = loss_fn(linear(features), labels)
            optimizer.zero_grad(); loss.backward(); optimizer.step()
    linear.eval()
    correct = total = 0
    with torch.no_grad():
        for images, labels in test_loader:
            images, labels = images.to(DEVICE), labels.to(DEVICE)
            correct += (linear(model.encode(images)).argmax(1) == labels).sum().item()
            total   += len(labels)
    for p in model.backbone.parameters(): p.requires_grad = True
    return correct / total

print("Evaluation utilities defined.")

## 4. Run All Batch Size Experiments

Each batch size gets its own SSL training run from scratch.
The eval loaders are shared and always use `EVAL_BATCH_SIZE` — batch size doesn't affect evaluation results.

In [ ]:
eval_train_loader, eval_test_loader = get_eval_loaders(EVAL_BATCH_SIZE)

all_histories = {}

for batch_size in BATCH_SIZES:
    ssl_loader = get_ssl_loader(batch_size)
    model      = SimCLRModel().to(DEVICE)
    criterion  = NTXentLoss(temperature=TEMPERATURE)
    optimizer  = Adam(model.parameters(), lr=SSL_LR, weight_decay=SSL_WEIGHT_DECAY)
    history    = {'loss': [], 'knn_acc': []}

    with _Tee(LOG_FILE):
        print(f"\n{'#'*60}")
        print(f"  Training: batch size = {batch_size}  "
              f"({2*batch_size - 2} negatives per image)")
        print(f"{'#'*60}")

        for epoch in range(1, SSL_EPOCHS + 1):
            model.train()
            total_loss = 0.0
            for (view1, view2), _ in ssl_loader:
                view1, view2 = view1.to(DEVICE), view2.to(DEVICE)
                _, z_i = model(view1)
                _, z_j = model(view2)
                loss   = criterion(z_i, z_j)
                optimizer.zero_grad(); loss.backward(); optimizer.step()
                total_loss += loss.item()

            avg_loss = total_loss / len(ssl_loader)
            history['loss'].append(avg_loss)

            run_knn = (epoch % KNN_INTERVAL == 0)
            if run_knn:
                acc = knn_monitor(model, eval_train_loader, eval_test_loader)
                history['knn_acc'].append((epoch, acc))

            if epoch % 10 == 0 or epoch == 1:
                knn_str = f"  kNN: {acc*100:.2f}%" if run_knn and epoch % 10 == 0 else ""
                print(f"  Epoch {epoch:3d}/{SSL_EPOCHS}  loss={avg_loss:.4f}{knn_str}")

        ckpt_path = f'{BASE_DIR}/checkpoints/simclr_bs{batch_size}.pt'
        torch.save(model.state_dict(), ckpt_path)
        print(f"  Saved: {ckpt_path}")

    all_histories[batch_size] = history

print(f"\nAll {len(BATCH_SIZES)} runs complete.")

## 5. Comparison Plots

In [ ]:
COLORS = ['#e63946', '#2a9d8f', '#e9c46a', '#457b9d', '#f4a261']

fig, axes = plt.subplots(1, 2, figsize=(13, 5))
fig.suptitle('SimCLR Batch Size Ablation Study', fontsize=14, fontweight='bold')

for i, bs in enumerate(BATCH_SIZES):
    history = all_histories[bs]
    color   = COLORS[i % len(COLORS)]
    label   = f'batch {bs}  ({2*bs-2} neg)'

    axes[0].plot(range(1, len(history['loss']) + 1), history['loss'],
                 color=color, label=label, linewidth=1.8)

    if history['knn_acc']:
        epochs, accs = zip(*history['knn_acc'])
        axes[1].plot(epochs, [a * 100 for a in accs],
                     color=color, label=label, linewidth=1.8,
                     marker='o', markersize=3)

axes[0].set_xlabel('Epoch');         axes[0].set_ylabel('NT-Xent Loss')
axes[0].set_title('Training Loss');  axes[0].legend(fontsize=8); axes[0].grid(alpha=0.3)

axes[1].set_xlabel('Epoch');         axes[1].set_ylabel('kNN Accuracy (%)')
axes[1].set_title(f'kNN Monitor (k={KNN_K})');
axes[1].legend(fontsize=8);          axes[1].grid(alpha=0.3)

plt.tight_layout()
plot_path = f'{BASE_DIR}/results/batchsize_ablation.png'
plt.savefig(plot_path, dpi=150)
plt.show()
print(f"Saved: {plot_path}")

## 6. Linear Probing Comparison

In [ ]:
lp_results = {}

for bs in BATCH_SIZES:
    model = SimCLRModel().to(DEVICE)
    model.load_state_dict(torch.load(f'{BASE_DIR}/checkpoints/simclr_bs{bs}.pt',
                                      map_location=DEVICE, weights_only=True))
    with _Tee(LOG_FILE):
        print(f"\n--- Linear Probing: batch size = {bs} ---")
        acc = linear_probing(model, eval_train_loader, eval_test_loader)
        print(f"    Accuracy: {acc*100:.2f}%")
    lp_results[bs] = acc

with _Tee(LOG_FILE):
    print(f"\n{'='*55}")
    print(f"  Batch Size Ablation — Final Results")
    print(f"{'='*55}")
    print(f"  {'Batch Size':>12}  {'Negatives/img':>15}  {'Linear Probe Acc':>18}")
    print(f"  {'-'*12}  {'-'*15}  {'-'*18}")
    for bs in BATCH_SIZES:
        print(f"  {bs:>12}  {2*bs-2:>15}  {lp_results[bs]*100:>17.2f}%")
    print(f"{'='*55}")

with open(f'{BASE_DIR}/results/batchsize_summary.txt', 'w') as f:
    f.write(f"Batch Size Ablation Study\n")
    f.write(f"SSL epochs: {SSL_EPOCHS}, temperature: {TEMPERATURE}\n\n")
    f.write(f"{'Batch Size':>12}  {'Negatives/img':>15}  {'Linear Probe Acc':>18}\n")
    for bs in BATCH_SIZES:
        f.write(f"{bs:>12}  {2*bs-2:>15}  {lp_results[bs]*100:>17.2f}%\n")
print(f"Summary saved to {BASE_DIR}/results/batchsize_summary.txt")

## Notes for your report

**What to look for and discuss:**

- **Large batch (512):** Many negatives per image — the model is forced to make fine-grained distinctions. kNN accuracy should rise faster and reach a higher ceiling.
- **Small batch (32):** Only 62 negatives — the task is easy and the learning signal is weak. kNN accuracy improves slowly or plateaus at a lower value.
- **The loss curves will look misleading:** A small batch run may show a *lower* loss value than a large batch run, because with fewer negatives the "classification" problem is easier. But the kNN accuracy tells the opposite story. This is a key point for your report — loss and representation quality are not the same thing in SSL.
- **Practical takeaway:** For SSL, always use the largest batch size your GPU allows.